# `features` -- Team-Level Feature Engineering

Two families of features live here:

1. **Scalar helpers** for delivery-level ("dynamic"/in-game)    features -- pure arithmetic functions, no state, used inline    throughout `pipeline.py`'s dynamic 2nd/1st-innings feature    construction.
2. **Match-level walk-forward features** -- `form` (recent win    rate) and `h2h` (head-to-head record) -- which are inherently    *stateful*: they need to remember every team's history as they    iterate through matches in chronological order.

**The single most important invariant in this whole module**: every walk-forward function only ever looks at matches *before* the one currently being processed. This is what lets these features be used honestly for prediction -- verified directly by `tests/test_features.py::test_form_walk_forward_no_leakage` and `test_h2h_no_lookahead`.

In [1]:
import numpy as np
import pandas as pd

## Scalar helpers for dynamic (in-game) features

These four functions are the building blocks of every delivery-level feature in `pipeline.py` (`build_dynamic_2nd`, `build_dynamic_1st`) and are reused, in spirit, by `game_state.py`'s more elaborate per-ball vector. Each one takes raw scoreboard numbers and returns a single derived quantity a human commentator would recognise immediately.

- **`balls_remaining`**: how many legal deliveries are left in a   20-over (120-ball) innings. Clipped to a minimum of 1 so   formulas that divide by it (like `rrr` below) never divide by   zero at the very last ball.
- **`runs_needed`**: for the chasing team, how many more runs are   required. Clipped to 0 once the target is passed (can't need   negative runs).
- **`crr`** (current run rate): runs scored per over so far --   the standard cricket-broadcast statistic.
- **`rrr`** (required run rate): the run rate the chasing team   needs to *maintain* to win -- this is the number that creates   match tension ("they need 9 an over from here").
- **`phase`**: which third of the innings a ball belongs to.   `over > 6` catches the Powerplay/Middle boundary and `over > 15`   catches Middle/Death; using two independent comparisons summed   together is a compact way to get a clean 0/1/2 without an   if/elif chain. Verified boundary-exact by   `tests/test_features.py::test_phase_powerplay/middle/death` and,   for the notebook-based `game_state` module,   `tests/test_game_state.py::test_phase_boundaries_exact`.
- **`get_enc`**: a small helper for target-encoding lookups   (mapping a category, like a venue, to a learned numeric value)   with a safe fallback to the global mean for unseen categories   -- avoids `KeyError` on a venue that never appeared in training   data.

In [2]:
def balls_remaining(team_balls: int, total_balls: int = 120) -> int:
    """Balls left in the innings; clipped to at least 1 to avoid division by zero."""
    return max(total_balls - team_balls, 1)

In [3]:
def runs_needed(target: int, team_runs: int) -> int:
    """Runs the chasing team still needs; clipped to 0 once target is passed."""
    return max(target - team_runs, 0)

In [4]:
def crr(team_runs: float, team_balls: float) -> float:
    """Current run rate (runs per over) from team-level ball-by-ball state."""
    return team_runs / max(team_balls, 1) * 6

In [5]:
def rrr(runs_needed_val: float, balls_remaining_val: float) -> float:
    """Required run rate (runs per over) for the chasing team."""
    return runs_needed_val / max(balls_remaining_val, 1) * 6

In [6]:
def phase(over: int) -> int:
    """
    Match phase based on over number (0-indexed).

    Returns
    -------
    0 : Powerplay  (overs 0–6)
    1 : Middle     (overs 7–15)
    2 : Death      (overs 16–19)
    """
    return int(over > 6) + int(over > 15)

In [7]:
def get_enc(enc_map: dict, global_mean: float, name: str) -> float:
    """Target-encoding lookup with fallback to global mean for unseen values."""
    return enc_map.get(name, global_mean)

## `compute_form_h2h()` -- rolling form + raw head-to-head

Iterates through every match **in chronological order**, maintaining two running dictionaries as it goes:

- `tw` ("team wins"): for every team, a list of 1/0 results   from their past matches (1 = won, 0 = lost), in order.
- `h2h`: for every unordered pair of teams (`frozenset({t1,t2})`   so `(A,B)` and `(B,A)` are the same key), a running tally of   how many times each side has won when they've met.

For each match, **before** updating either dictionary with this match's result, it reads off:
- `form1`/`form2`: team1/team2's win rate in their last `window`   (default 5) matches, or exactly `0.5` (a neutral prior) if the   team has no history yet -- this avoids an undefined   average-of-nothing for each team's very first appearance.
- `h2h_rate`: team1's historical win rate specifically against   team2, again defaulting to `0.5` with no prior meetings.

Only *after* recording these values does it update `tw`/`h2h` with the actual outcome of the current match -- this ordering is exactly what prevents lookahead leakage.

It also returns the final `tw`/`h2h` dictionaries themselves (not just the per-row feature lists), because `pipeline.py`'s 2026 external evaluation needs to *continue* this walk-forward process into the future, picking up exactly where the historical data left off.

In [8]:
def compute_form_h2h(
    match_df: pd.DataFrame,
    window: int = 5,
) -> tuple[list, list, list, dict, dict]:
    """
    Walk-forward computation of recent team form and head-to-head rate.

    Iterates through matches in order; each match sees only results from
    strictly prior matches (no lookahead).

    Parameters
    ----------
    match_df : pd.DataFrame
        Must contain columns: team1, team2, team1_win.
        Must already be sorted chronologically.
    window : int
        Number of recent matches used for form computation.

    Returns
    -------
    form1, form2 : list[float]
        Rolling win rate for team1 / team2 in their last `window` matches.
    h2h_rate : list[float]
        Raw head-to-head win rate for team1 vs team2 (default 0.5 if no history).
    tw : dict
        Accumulated per-team result history (used for walk-forward 2026 eval).
    h2h : dict
        Accumulated per-pair H2H counts (used for walk-forward 2026 eval).
    """
    tw: dict = {}
    h2h: dict = {}
    form1, form2, h2h_rate = [], [], []

    for _, r in match_df.iterrows():
        t1, t2, out = r["team1"], r["team2"], r["team1_win"]
        h1 = tw.get(t1, [])
        h2 = tw.get(t2, [])
        form1.append(np.mean(h1[-window:]) if h1 else 0.5)
        form2.append(np.mean(h2[-window:]) if h2 else 0.5)
        key = frozenset([t1, t2])
        if key not in h2h:
            h2h[key] = {t1: 0, t2: 0, "n": 0}
        e = h2h[key]
        h2h_rate.append(e[t1] / e["n"] if e["n"] > 0 else 0.5)
        tw.setdefault(t1, []).append(out)
        tw.setdefault(t2, []).append(1 - out)
        e[t1 if out else t2] += 1
        e["n"] += 1

    return form1, form2, h2h_rate, tw, h2h

## `compute_h2h_beta()` -- Bayesian-smoothed head-to-head

**Why this exists:** the raw `h2h_rate` above is unreliable when two teams have only played once or twice -- a single upset gives a raw win rate of 0% or 100%, which is a wildly overconfident signal to hand to a model.

This function computes the **same running head-to-head tally** as `compute_form_h2h`, but instead of the raw ratio `wins/n`, it uses a **Beta(2,2) prior**: `(wins + alpha) / (n + alpha + beta)`. With `alpha=beta=2` this is equivalent to pretending each side already has 2 "phantom" wins before any real matches are counted, which pulls small-sample estimates toward the neutral 0.5 point while barely affecting the estimate once there's a real sample size (e.g. 1 win from 1 match -> 75% instead of 100%; 3 wins from 5 -> 56% instead of 60%).

Ablation in `project_gagan`'s original notebook (cell 14) showed this beats the raw H2H feature: BSS improved from -0.0516 to -0.0390 on the internal 2021-2025 test set (still negative overall -- pre-match prediction has a low ceiling -- but meaningfully less bad).

In [9]:
def compute_h2h_beta(
    match_df: pd.DataFrame,
    alpha: float = 2.0,
    beta_param: float = 2.0,
) -> list[float]:
    """
    Bayesian Beta-prior H2H win rate (Papers 47, 51).

    Raw H2H win rates are unreliable for sparse matchups. A symmetric
    Beta(alpha, beta_param) prior shrinks extreme rates toward 0.5.
    With alpha=beta_param=2, the posterior mean is (wins+2)/(n+4).

    Computed in match order with no lookahead.
    """
    h2h: dict = {}
    col = []
    for _, r in match_df.iterrows():
        t1, t2 = r["team1"], r["team2"]
        key = frozenset([t1, t2])
        if key not in h2h:
            h2h[key] = {t1: 0, t2: 0, "n": 0}
        e = h2h[key]
        col.append((e[t1] + alpha) / (e["n"] + alpha + beta_param))
        out = r["team1_win"]
        e[t1 if out else t2] += 1
        e["n"] += 1
    return col